# Identifying Problem Stocks
In this notebook I will identify potential outlier stocks that show abnormal return 
trends with disproportional risk.

Black swan events such as the 2008 financial crisis, COVID-19, the dotcom bubble burst,
GameStop bull run may be legitimate reasons for outlier returns.

However, sometimes financial data may be corrupted for illegitimate reasons including,
double counting, acquisitions & mergers, bankruptcies etc.

It is important to identify potentially corrupted data so that it does not intefere with
algorithm testing and signal generation.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import random
from matplotlib import pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import date
from dateutil.relativedelta import relativedelta

from trading_algos import optimization as tao
from trading_algos import datasets as tad
from trading_algos import plots as tap
from trading_algos import utils as tau
from trading_algos.utils import head_tail as ht

%load_ext autoreload
%autoreload 2

## Load Data

In [ ]:
# Load a complete collection of S&P500 stocks from repository
# Not interested in survivors here
df_stocks = tad.get_sp500(survivors=False)

In [ ]:
ht(df_stocks)

In [ ]:
df_stocks.shape

In [ ]:
print(df_stocks.columns.get_level_values(1).nunique(), 'stocks')

## Identifying Outliers

In [ ]:
log_prices = np.log(df_stocks.Close)
ht(log_prices)

In [ ]:
# 3 Month Rolling Returns
rolling_log_return = log_prices - log_prices.shift(63)
ht(rolling_log_return)

In [ ]:
df_agg = rolling_log_return.agg({'mean', 'std'}).rename({'mean': 'Return', 'std': 'Risk'}).T
ht(df_agg)

In [ ]:
lq = df_agg.quantile(0.25)
uq = df_agg.quantile(0.75)
iqr = uq - lq
lo = lq - 3*iqr
uo = uq + 3*iqr

In [ ]:
df_outliers = df_agg[((df_agg > uo) | (df_agg < lo)).all(axis=1)].sort_values(['Risk', 'Return'], ascending=False)
outliers = df_outliers.index.to_list()
df_outliers.shape

In [ ]:
df_outliers.head()

In [ ]:
outliers

In [ ]:
top_ret = abs(df_agg['Return']).sort_values(ascending=False).index[:10]
top_rsk = abs(df_agg['Risk']).sort_values(ascending=False).index[:10]

In [ ]:
df_agg.loc[top_ret, 'Return']

In [ ]:
df_agg.loc[top_rsk, 'Risk']

In [ ]:
outliers

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(12,6), tight_layout=True)

ax[0].set_title('Risk')
ax[0].spines[['left','top','right']].set_visible(False)
sns.boxplot(df_agg[['Risk']], ax=ax[0], orient='h')

ax[1].set_title('Return')
ax[1].spines[['left','top','right']].set_visible(False)
sns.boxplot(df_agg[['Return']], ax=ax[1], orient='h')

In [ ]:
ncols = 5
nrows = int(np.ceil(len(outliers)/ncols))


fig, axes = plt.subplots(nrows, ncols, figsize=(12, 30), tight_layout=True)

fig.tight_layout(pad=2, rect=[0,0.05,1,0.95])
fig.suptitle(f'Log Price Trend of Outlier Stocks', fontsize=16)

for i, ticker in enumerate(outliers):

    ax = axes[int(i/5), i%5]
    ax.set_title(f"{ticker}")
    ax.set_yticklabels([])
    ax.set_yticks([])
    ax.set_ylabel(' ')
    ax.set_xticklabels([])
    ax.set_xticks([])
    ax.set_xlabel(' ')
    ax.spines[['top','bottom','left','right']].set_visible(False)

    ax.plot((log_prices[ticker]), linewidth=0.5)

# Determine how many empty plots there are on the last row of the figure
num_empty_plots = nrows * ncols - len(outliers)
if num_empty_plots > 0:
    for i in range(1, num_empty_plots + 1):
        # Hide empty plots
        axes[nrows - 1, ncols - i].set_axis_off()

In [ ]:
# For the sake of this notebook I will drop all outliers
# Usually would investigate further to decide which ones need dropping
log_prices = log_prices.drop(columns=outliers)

In [ ]:
df_agg = df_agg.drop(outliers)

In [ ]:
fig, ax = plt.subplots(1,2)

sns.boxplot(df_agg[['Risk']], ax=ax[0])
sns.boxplot(df_agg[['Return']], ax=ax[1])

In [ ]:
log_returns = log_prices - log_prices.shift(1)
rolling_median = log_returns.rolling(21).median()

In [ ]:
def calc_mad(x):

    print(x)
    median = np.median(x)
    print(median)
    print(np.median(np.abs(x - median)))
    return np.median(np.abs(x - median))

In [ ]:
rolling_mad = log_returns.rolling(21).apply(calc_mad, raw=True)

In [ ]:
rolling_mad = rolling_mad.replace(0, 1e-8)

In [ ]:
df_z = (0.6745 * (log_returns - rolling_median) / rolling_mad)

In [ ]:
(abs(df_z) > 3.5).sum().sort_values(ascending=False).head(20)

In [ ]:
log_prices['EP'].plot()

In [ ]:
rolling_median['EP'].plot()

In [ ]:
df_z['EP'].plot()

In [ ]:
log_prices['TIE'].plot()

In [ ]:
(log_returns - rolling_median) / rolling_mad

In [ ]:
log_prices['AA'].plot()

In [ ]:
df_z.max().sort_values(ascending=False).head(20)

In [ ]:
rolling_median